# SystemDesignExpertLLM — DPO refinement on Kaggle

Loads the SFT LoRA adapter and refines it with DPO on preference pairs
(`chosen` = teacher answer, `rejected` = base-model answer).

**Prereqs:** run `kaggle_sft.ipynb` first and attach its `adapters/sft` output, plus the
`dpo.jsonl` dataset. Set `SFT_ADAPTER` and `DPO_PATH` below.

In [ ]:
%%capture
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
import os
BASE_MODEL  = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 4096
SFT_ADAPTER = "/kaggle/input/sdx-sft-adapter/sft"   # <-- attach kaggle_sft.ipynb output
DPO_PATH    = "/kaggle/input/sdx-dpo/dpo.jsonl"      # <-- attach dpo dataset
OUT_ADAPTER = "/kaggle/working/adapters/dpo"
OUT_MERGED  = "/kaggle/working/merged/dpo-fp16"
BETA        = 0.1
LR          = 5e-6
EPOCHS      = 1
SEED        = 3407
os.makedirs(OUT_ADAPTER, exist_ok=True)

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()   # must be called before importing the DPO trainer
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
# resume from the SFT adapter, then keep training its LoRA weights
model.load_adapter(SFT_ADAPTER, adapter_name="default")
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a distinguished software architect. Given a real requirement, you produce "
    "concrete, production-grade system design guidance with explicit technology choices "
    "and rigorous tradeoff analysis."
)

def fmt(ex):
    prompt = tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":ex["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {"prompt": prompt, "chosen": ex["chosen"], "rejected": ex["rejected"]}

ds = load_dataset("json", data_files=DPO_PATH, split="train").map(fmt)
print(ds)

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model = model,
    ref_model = None,           # Unsloth/PEFT uses the base as implicit reference
    tokenizer = tokenizer,
    train_dataset = ds,
    args = DPOConfig(
        beta = BETA,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.1,
        logging_steps = 10,
        save_steps = 100,
        optim = "adamw_8bit",
        seed = SEED,
        max_length = MAX_SEQ_LEN,
        max_prompt_length = 1024,
        output_dir = OUT_ADAPTER,
        report_to = "none",
    ),
)
trainer.train()

In [ ]:
model.save_pretrained(OUT_ADAPTER); tokenizer.save_pretrained(OUT_ADAPTER)
model.save_pretrained_merged(OUT_MERGED, tokenizer, save_method="merged_16bit")
print("DPO adapter ->", OUT_ADAPTER, "| merged ->", OUT_MERGED)